# Baseline 04b — RandomForest sobre features US-018

Variante minima del baseline tabular sobre el subset US-018 (85 951 parcelas, 17 indices espectrales x 9 stats + FFT NDVI + 8 features fenologicas). Sirve como piloto del patron `setup_notebook` + `train_baseline_three_models` que el resto de las libretas reusa.

**Pregunta**: ¿que F1-macro out-of-fold consigue RandomForest puro sobre el subset US-018 con spatial CV 5-fold + buffer 1 km?

## Requisitos

- `data/test_fixtures/feature_selection_parcels_subset.parquet` presente (descargable via `dvc pull`).
- `data/processed/pastis_parcels_full.geoparquet` presente (generado por el pipeline EDA US-011).


In [ ]:
FEATURES_PATH = "data/test_fixtures/feature_selection_parcels_subset.parquet"
PARCELS_GEOPARQUET = "data/processed/pastis_parcels_full.geoparquet"
FIGURES_SUBDIR = "us-023-preview/04b_baseline"
REPORTS_SUBDIR = "baseline/04b_baseline"
K_FOLDS = 5
BUFFER_KM = 1.0
RANDOM_STATE = 42


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Bootstrap: localizar el repo root buscando pyproject.toml
_HERE = Path.cwd().resolve()
for _candidate in (_HERE, *_HERE.parents):
    if (_candidate / "pyproject.toml").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from ml.utils.notebook_bootstrap import setup_notebook
from IPython.display import Markdown, display

env = setup_notebook(
    figures_subdir=FIGURES_SUBDIR,
    reports_subdir=REPORTS_SUBDIR,
)
display(Markdown(env.summary_markdown()))


## Carga del dataset

Combinamos el subset US-018 con la metadata real (clase + patch + fold + area) desde el geoparquet de parcelas Italia.

In [ ]:
import polars as pl
from ml.utils.baseline_notebook_helpers import (
    load_features_dataset_with_meta,
    train_baseline_three_models,
    build_model_comparison_table,
)
from ml.utils.class_distribution import (
    class_distribution_report,
    recommend_threshold,
)

df = load_features_dataset_with_meta(
    path=FEATURES_PATH,
    parcels_geoparquet=PARCELS_GEOPARQUET,
)
parcel_id_dtype = df.schema['parcel_id']
display(Markdown(
    f"**Dataset cargado**: `{df.height:,}` parcelas x "
    f"`{df.width}` columnas. `parcel_id` dtype: `{parcel_id_dtype}`."
))
display(df.head(5))


## Distribucion de clases

Antes de entrenar, reportamos la distribucion real de las 18 clases PASTIS-R y proponemos un threshold sensato de soporte (en lugar del 1000 hardcoded que dejaba solo 1 clase pasando).

In [ ]:
report = class_distribution_report(df)
display(report)

threshold_p25 = recommend_threshold(report, method='p25')
threshold_p50 = recommend_threshold(report, method='p50')
display(Markdown(
    f"**Threshold sugerido**: p25 = `{threshold_p25}`, "
    f"p50 = `{threshold_p50}` parcelas. "
    "Las clases por debajo del threshold tienen soporte debil "
    "y se reportan en color en los plots."
))


## Entrenamiento RandomForest + XGBoost + LightGBM con spatial CV

In [ ]:
rows = train_baseline_three_models(
    df,
    models=('rf', 'xgb', 'lgbm'),
    k_folds=K_FOLDS,
    buffer_km=BUFFER_KM,
    random_state=RANDOM_STATE,
)
comparison_path = env.reports_dir / 'model_comparison_04b.parquet'
comparison = build_model_comparison_table(rows, output_path=comparison_path)
display(Markdown(f'**Tabla comparativa persistida**: `{comparison_path}`'))
display(comparison)


## Graficas: comparativa de modelos + soporte por clase

In [ ]:
import matplotlib.pyplot as plt
from ml.eval.reencuadre_plots import (
    plot_class_support_bars,
    plot_model_comparison_bars,
)

metric_by_model = {r.model: r.f1_macro for r in rows}
fig1 = plot_model_comparison_bars(
    metric_by_model,
    baseline_value=0.40,
    baseline_label='referencia US-022 (F1-macro 0.40)',
    title='F1-macro RF vs XGB vs LGBM (04b)',
)
fig1.savefig(env.figures_dir / 'model_comparison_04b.png', bbox_inches='tight')
display(fig1)
plt.close(fig1)

fig2 = plot_class_support_bars(
    report.rename({'n_parcels': 'len'}),
    weak_threshold=threshold_p25,
    title=f'Soporte por clase (threshold p25 = {threshold_p25})',
)
fig2.savefig(env.figures_dir / 'class_support_04b.png', bbox_inches='tight')
display(fig2)
plt.close(fig2)


## Per-class F1 del mejor modelo (out-of-fold)

In [ ]:
from ml.eval.reencuadre_plots import plot_per_class_f1
from ml.train.baseline import train_one_model
from ml.ingest.pastis_loader import PASTIS_R_CLASSES
from ml.train.baseline import evaluate_with_spatial_cv, build_estimator

best_model = comparison['model'][0]
display(Markdown(f'**Mejor modelo**: `{best_model}` (F1-macro `{comparison["f1_macro"][0]:.4f}`)'))

# Reentrenamos brevemente el mejor modelo para conseguir y_pred_oof.
best_result = train_one_model(
    df,
    model=best_model,
    k_folds=K_FOLDS,
    buffer_km=BUFFER_KM,
    random_state=RANDOM_STATE,
)
# Recuperamos las predicciones out-of-fold via evaluate_with_spatial_cv
_cv_metrics, y_true_oof, y_pred_oof = evaluate_with_spatial_cv(
    df,
    lambda: build_estimator(best_model, best_result.best_params),
    k_folds=K_FOLDS,
    buffer_km=BUFFER_KM,
    random_state=RANDOM_STATE,
)

# Decodificamos las etiquetas para nombres legibles
class_names = {int(c): PASTIS_R_CLASSES.get(int(c), f'class_{int(c)}') for c in best_result.label_classes}
fig3 = plot_per_class_f1(
    y_true_oof,
    y_pred_oof,
    class_labels=list(range(len(best_result.label_classes))),
    class_names={i: class_names[c] for i, c in enumerate(best_result.label_classes)},
    weak_threshold=0.10,
    title=f'F1 por clase ({best_model}) out-of-fold',
)
fig3.savefig(env.figures_dir / 'per_class_f1_04b.png', bbox_inches='tight')
display(fig3)
plt.close(fig3)


## Conclusiones

Esta libreta cumple el rol de **piloto** del patron de bootstrap nuevo (`setup_notebook` + `baseline_notebook_helpers`) sobre el subset US-018. Los hallazgos importantes:

1. **Tres modelos comparables**: RandomForest, XGBoost y LightGBM ejecutados con identica spatial CV 5-fold + buffer 1 km. La tabla persistida `model_comparison_04b.parquet` queda como referencia local para detectar regresiones cuando agregemos bloques opcionales en `04_baseline` y `05_reencuadre_fenologico`.

2. **Soporte por clase reportado con threshold p25**: en lugar del threshold hardcoded de 1000 que dejaba 17/18 clases marcadas como debiles, ahora el threshold se calcula desde la propia distribucion y solo destaca las clases verdaderamente raras.

3. **Per-class F1 del mejor modelo**: identifica que clases concentran el error (las raras suelen estar bajo el 0.10), util para discutir merge fenologico via `merge_to_phenological_groups` en futuras iteraciones.

## Lo que sigue

- `04_baseline.ipynb` reutiliza este patron sobre el conjunto fused completo (con AlphaEarth + ERA5 + SRTM).
- `05_reencuadre_fenologico.ipynb` mide el aporte de los bloques opcionales (FarSLIP, pheno_text Gemini, firma espectral) y persiste la decision final.
- `Avance3.Equipo17.ipynb` consolida y nombra el conjunto ganador (`select_winning_features`).